In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['AmazonLaptops'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [7]:
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza completada.")

NOMBRE_GRUPO = "Data_traders"
datos_finales = []
driver = None

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado.")
    driver.get("https://www.amazon.es/s?k=laptops")

    limite_paginas = 3

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        time.sleep(5)

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )
        print("Bloques encontrados:", len(bloques))

        for i, bloque in enumerate(bloques):
            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, "h2 span").text.strip()

                precio_elementos = bloque.find_elements(By.CSS_SELECTOR, ".a-price .a-offscreen")
                if not precio_elementos:
                    print(f"Bloque {i}: sin precio")
                    continue

                precio_texto = precio_elementos[0].get_attribute("innerHTML").strip()
                precio_limpio = (
                    precio_texto.replace("€", "")
                    .replace(".", "")
                    .replace(",", ".")
                    .strip()
                )

                datos_finales.append({
                    "identificador": nombre,
                    "valor": float(precio_limpio),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })

            except Exception as e:
                print(f"Bloque {i}: error -> {e}")
                continue

        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(3)
        except Exception as e:
            print("No se encontró siguiente página:", e)
            break

    print(f"Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f"Error general en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")
    

Limpieza completada.
Navegador iniciado.
--- Procesando Página 1 ---
Bloques encontrados: 60
Bloque 3: sin precio
Bloque 12: sin precio
Bloque 14: sin precio
Bloque 20: sin precio
Bloque 22: sin precio
Bloque 24: sin precio
Bloque 30: sin precio
Bloque 31: sin precio
Bloque 32: sin precio
Bloque 46: sin precio
Bloque 49: sin precio
Bloque 50: sin precio
Bloque 53: sin precio
Bloque 57: sin precio
--- Procesando Página 2 ---
Bloques encontrados: 60
Bloque 4: sin precio
Bloque 5: sin precio
Bloque 6: sin precio
Bloque 8: sin precio
Bloque 12: sin precio
Bloque 14: sin precio
Bloque 20: sin precio
Bloque 24: sin precio
Bloque 25: sin precio
Bloque 26: sin precio
Bloque 27: sin precio
Bloque 29: sin precio
Bloque 30: sin precio
Bloque 37: sin precio
Bloque 43: sin precio
Bloque 45: sin precio
Bloque 46: sin precio
Bloque 47: sin precio
Bloque 49: sin precio
Bloque 51: sin precio
Bloque 56: sin precio
Bloque 58: sin precio
--- Procesando Página 3 ---
Bloques encontrados: 60
Bloque 9: sin pr

In [13]:
# ==========================================
# IMPORTS Y LIMPIEZA
# ==========================================
import os
import time
from datetime import datetime
from zoneinfo import ZoneInfo

from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Limpieza de procesos anteriores
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada.")

# ==========================================
# CONFIGURACIÓN GENERAL
# ==========================================
HOST_MONGO = "mongodb"   # Si falla, cambia por "database"
PUERTO_MONGO = 27017

BASE_DATOS = "proyecto_bigdata"
COLECCION = "datos_finanzas_yahoo_20260415"   # Colección nueva

NOMBRE_GRUPO = "Data_Traders"

URLS = {
    "gainers": "https://finance.yahoo.com/markets/stocks/gainers/",
    "losers": "https://finance.yahoo.com/markets/stocks/losers/",
    "most_active": "https://finance.yahoo.com/markets/stocks/most-active/"
}

datos_finales = []

# ==========================================
# FUNCIÓN PARA LIMPIAR NÚMEROS
# ==========================================
def limpiar_numero(texto):
    if not texto:
        return None

    texto = str(texto).strip()

    if texto in ["", "-", "--", "N/A"]:
        return None

    # Quita símbolos comunes
    texto = texto.replace(",", "")
    texto = texto.replace("%", "")
    texto = texto.replace("+", "")
    texto = texto.replace("$", "")
    texto = texto.strip()

    try:
        return float(texto)
    except:
        return None

# ==========================================
# FUNCIÓN FECHA CHILE
# ==========================================
def fecha_chile():
    return datetime.now(ZoneInfo("America/Santiago")).strftime("%Y-%m-%d %H:%M:%S")

# ==========================================
# CONFIGURACIÓN SELENIUM
# ==========================================
options = Options()
options.binary_location = "/usr/bin/google-chrome"

# Si quieres verlo en VNC, deja comentado headless
# options.add_argument("--headless")

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = None

# ==========================================
# SCRAPING
# ==========================================
try:
    driver = webdriver.Chrome(options=options)
    print("🌐 Navegador iniciado correctamente.")

    for categoria, url in URLS.items():
        print(f"\n📊 Procesando categoría: {categoria}")
        driver.get(url)
        time.sleep(8)

        filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        print(f"Filas encontradas en {categoria}: {len(filas)}")

        for i, fila in enumerate(filas):
            try:
                lineas = [x.strip() for x in fila.text.split("\n") if x.strip()]

                # Validación mínima
                if len(lineas) < 5:
                    print(f"Fila {i}: muy corta -> {lineas}")
                    continue

                simbolo = lineas[0]
                nombre = lineas[1]
                precio = limpiar_numero(lineas[2])

                cambio = None
                cambio_pct = None

                if len(lineas) > 3:
                    partes = lineas[3].split()
                    if len(partes) >= 1:
                        cambio = limpiar_numero(partes[0])
                    if len(partes) >= 2:
                        cambio_pct = limpiar_numero(partes[1])

                volumen = lineas[4] if len(lineas) > 4 else None

                registro = {
                    "item": nombre,
                    "simbolo": simbolo,
                    "valor": precio,
                    "cambio": cambio,
                    "cambio_porcentual": cambio_pct,
                    "volumen": volumen,
                    "categoria": categoria,
                    "grupo": NOMBRE_GRUPO,
                    "fecha_captura": fecha_chile()
                }

                if registro["item"] and registro["valor"] is not None:
                    datos_finales.append(registro)

            except Exception as e:
                print(f"Fila {i}: error -> {e}")

    print(f"\n✅ Extracción terminada: {len(datos_finales)} registros.")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# ==========================================
# GUARDAR EN MONGODB
# ==========================================
try:
    client = MongoClient(HOST_MONGO, PUERTO_MONGO, serverSelectionTimeoutMS=5000)

    # Verifica conexión real
    client.admin.command("ping")
    print("✅ Conexión exitosa a MongoDB.")

    db = client[BASE_DATOS]
    coleccion = db[COLECCION]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print("💾 Datos guardados correctamente en MongoDB.")
    else:
        print("⚠️ No hay datos para guardar.")

    # ==========================================
    # VERIFICACIÓN FINAL
    # ==========================================
    print("\n📂 Bases de datos disponibles:")
    print(client.list_database_names())

    print(f"\n📂 Colecciones en '{BASE_DATOS}':")
    print(db.list_collection_names())

    total = coleccion.count_documents({})
    print(f"\n📊 Total de documentos en '{COLECCION}': {total}")

    print(f"\n📄 Primeros documentos guardados en '{COLECCION}':")
    for doc in coleccion.find().limit(5):
        print(doc)

except Exception as e:
    print("❌ Error en MongoDB:", e)

🧹 Limpieza completada.
🌐 Navegador iniciado correctamente.

📊 Procesando categoría: gainers
Filas encontradas en gainers: 25

📊 Procesando categoría: losers
Filas encontradas en losers: 25

📊 Procesando categoría: most_active
Filas encontradas en most_active: 25

✅ Extracción terminada: 75 registros.
✅ Conexión exitosa a MongoDB.
💾 Datos guardados correctamente en MongoDB.

📂 Bases de datos disponibles:
['TiendaBigData', 'admin', 'config', 'local', 'proyecto_bigdata']

📂 Colecciones en 'proyecto_bigdata':
['datos_finanzas', 'datos_finanzas_yahoo_20260415', 'datos_scraping']

📊 Total de documentos en 'datos_finanzas_yahoo_20260415': 75

📄 Primeros documentos guardados en 'datos_finanzas_yahoo_20260415':
{'_id': ObjectId('69dfb6cd823a82219e0ae3fe'), 'item': 'Xanadu Quantum Technologies Limited', 'simbolo': 'XNDU', 'valor': 21.3, 'cambio': 6.47, 'cambio_porcentual': 43.63, 'volumen': '6.97', 'categoria': 'gainers', 'grupo': 'Data_Traders', 'fecha_captura': '2026-04-15 12:02:55'}
{'_id': O